# ASOC E4: independent constant-delay rerun

CPU only. This notebook writes the frozen sources, obtains pinned external releases, runs reserved checks, then calibrates and evaluates new fits with the recorded design. The package report discloses the earlier interrupted attempt. It is an adapted architecture/solver comparison, not a reproduction of the source paper’s original experiment. Run all cells in order. A full run can take substantial time; save outputs before a runtime ends.


In [ ]:
import sys,subprocess,json
from pathlib import Path
root=Path("/content/asoc_e4")
root.mkdir(parents=True,exist_ok=True)
source_files={'protocol.json': '{\n "protocol_id":"ASOC-E4-recovery-v1",\n "generator":{"growth":1.1,"capacity":1.5,"true_delay":0.28},\n "delay_bounds":[0.15,0.5],"initial_delay":0.325,\n "history_amplitude":0.18,"history_frequency":3.141592653589793,\n "history_c":[-1.0,0.0,1.0,0.0],"history_d":[0.0,1.0,0.0,-1.0],\n "train_states":[0.45,0.7,0.95,1.2],"validation_states":[0.55,0.8,1.05],"test_states":[0.575,0.825,1.075],\n "horizon":1.0,"observation_step":0.05,"fine_observation_step":0.025,"training_step":0.025,\n "evaluation_steps":[0.025,0.0125,0.00625],"reference_steps":[0.0025,0.00125,0.000625,0.0003125],"reference_tolerance":0.000001,\n "noise_levels":[0.0005,0.0015],"delay_profile_points":351,"slope_window_duration":0.2,"slope_polynomial_degree":2,"geometry_norm_threshold":0.00000001,\n "methods":["External-C","External-C+anchors","Structured-C","Structured-C+anchors"],\n "learning_rates":[0.0005,0.002],"schedules":{"280":[[0.5,90],[1.0,190]],"1200":[[0.5,400],[1.0,800]]},\n "anchor_weight":0.08,"field_weight_decay":0.000001,"gradient_clip":5.0,\n "calibration_data_seeds":[20261501,20261502],"calibration_model_seed":20261511,\n "final_data_seeds":[20261401,20261402,20261403,20261404,20261405],"optimization_seeds":[20261411,20261412,20261413],\n "workers":8,"threads_per_worker":1,\n "solver_commit":"4888c63dc84e966f6c6553277adc3851a2cca4fc","architecture_source_commit":"9b573c4bed5efee9fe9c6b765044dbd506b5bf07",\n "smoke":{"noise_levels":[0.0005],"learning_rates":[0.002],"schedules":{"4":[[0.5,2],[1.0,2]]},"calibration_data_seeds":[930001],"calibration_model_seed":930002,"final_data_seeds":[930101],"optimization_seeds":[930102]}\n}\n', 'PROTOCOL.md': "# ASOC E4 recovery protocol\n\nThis reconstructs the interrupted constant-delay comparison. Workspace maintenance removed its code, calibration records and partial final outputs before durable saving. Earlier progress messages showed 64 finite calibration fits and their selected settings; these observations are disclosed, not treated as recovered records. Re-run every trial and final fit. This is not a newly blind preregistration. Scientific design, seeds, objectives, selection rules and outcome definitions remain as previously agreed. Use eight one-thread CPU workers instead of four to reduce elapsed time; record actual runtime. Freeze these rebuilt sources before this rerun and save a durable package before fitting and at stage/chunk boundaries.\n\nThe generator, histories, state splits, numerical steps, noise conditions and seeds are exactly specified in protocol.json. x'=1.1x[1-x(t-0.28)/1.5]; 16 training/12 validation/12 test trajectories. All use four analytic trigonometric histories with coefficients c=(-1,0,1,0), d=(0,1,0,-1), amplitude .18 and frequency pi. Initial states/histories are exact. Fitting observations are .05 apart through time1. Gaussian noise only after zero; standardized draws generated on .025 grid and shared between noise levels. Split seed offsets train0, validation2000, test1000. Use float64 RK4 method-of-steps with linear delayed interpolation and the first passing consecutive reference-step pair (maximum difference <=1e-6 over all40 trajectories on .025 grid). Abort otherwise; this is empirical convergence, not a rigorous enclosure.\n\nCompute four anchors only from training observations: right-side degree2 polynomial on [0,.20] (five points), centered-history profiled least squares, 351 candidates [.15,.50], first grid minimizer; require centered signature norms >1e-8. Do not join independent trajectories or assume derivative compatibility across the zero/history junction. Anchor-C is the projected mean of four estimates; it has no trajectory prediction. Anchors reuse observations and add no independent labels.\n\nExternal-C: current/delayed scalar MLP, four hidden layers width32/ReLU, scalar output. Structured-C: width28/22 two-hidden-layer tanh networks, a(x)-(.05+softplus(b(x)))x_delayed. Unknown fields in both, but the structured class has affine-channel/sign prior. Each +anchors variant uses the same architecture/initialization with anchor loss weight .08. External-C+anchors is our adaptation. All learn one scalar parameter initialized .325 and projected in place to [.15,.50], preserving identity. Field decay1e-6, delay decay0; Adam; gradient clip5; weighted trajectory MSE time weights .6..1.4. Scalar-delay smoothness is identically zero.\n\nAll four use unchanged pinned torchdde Euler, direct backpropagation, fitting step.025 and analytic history/t0=0. Sources: Monsel et al., DOI10.1017/dce.2026.10053; architecture commit9b573c4bed5efee9fe9c6b765044dbd506b5bf07; solver commit4888c63dc84e966f6c6553277adc3851a2cca4fc. The archived adjoint route failed prior retained fixtures and is excluded. Revalidate direct-mode gradients in this environment. Common history, bounds, scalarizing our delay branch, Euler, fixed-rate Adam and curriculum differ from the native observed-prefix/adaptive/OneCycle experiment. This is an adapted architecture/solver comparison, not a reproduction of the source paper's original experiment. Do not modify or bundle external repos.\n\nCalibration:64 actual fits, four methods x two noise levels x rates[.0005,.002] x budgets[280,1200] x two separate noise datasets. Budget280 splits horizons.5/1 for90/190 updates;1200 splits400/800. Select per method/noise by mean noisy validation trajectory MSE at fitting step.025 across both datasets. Ties shorter budget, then smaller rate. Candidate eligible only if both fits are finite and complete. Retain completed failures without retries; if none eligible, mark unavailable and withhold final comparisons. No truth-based or test-based selection. Lock all choices/hash calibration records before final fit submission. The bounded grid does not establish optimal tuning.\n\nFinal: five new base-noise datasets, three optimizer repeats per method/noise cell, up to120 neural fits plus10 Anchor-C estimates. Actual model seed is optimizer seed+100 times dataset index. Fit from scratch, use the final update, no early stopping or checkpoint selection. Frozen evaluation on common .025 observation grid at Euler steps.025/.0125/.00625; clean held-out trajectory MSE at.00625 and scalar absolute delay error are separate primary outcomes. Retain all step-specific train/test clean/noisy MSEs, predicted observation minima, nonpositive observation counts, clipping/projection/boundary counts, parameter counts/hashes and elapsed times. No continuous-time positivity claim. Include unfavorable finite outcomes.\n\nSummarize medians of three optimizer repeats within each dataset, then all five dataset medians, marginal medians/ranges and paired ratios/differences. Within-architecture anchored/unanchored and cross-architecture pairs are predefined in e4_analysis.py. Ratios below1 favor numerator. If required repeats/datasets fail or are missing, withhold complete comparisons. Zero denominators have undefined ratios, retain differences. Noise conditions and optimizer repeats are not independent data replications. No population confidence interval, significance claim or new20% target. Separately tuned final pairs compare calibrated pipelines if chosen rates differ; they are not a pure fixed-setting intervention.\n\nReserved smoke uses seeds930001/930002 and930101/930102, one noise/rate, four updates; exclude it from evidence. Validate failure/missing/tie selection rules, gradients, pairing and projection identity, then smoke runner and replay. Freeze code/config/analysis/audit hashes before full rerun. Resume only identical sources/runtime; interrupted unfinished runs may restart, completed failures never retry. Atomic individual records recover parent interruptions. Save persistent snapshots before/after calibration and each final chunk.\n\nReplay every successful checkpoint, regenerate every raw dataset/anchor, independently recompute selection and metric arithmetic, verify frozen sources and external tracked hashes, verify configuration-lock timing and final cells, and recompute dataset/paired statistics. This is a consistency audit with shared solver/model code, not independent scientific replication. A constant-delay comparison does not establish superiority over an unknown state-dependent-delay learner. Preserve E1–E3 criteria and unfavorable outcomes; the manuscript is not submission-ready until integration and PDF checks.\n", 'e4_data.py': '"""Constant-delay data and observed-data anchors; truth stays outside fitting."""\nimport numpy as np\n\ndef metadata(states,c):\n return dict(s=np.repeat(np.array(states,float),4),c=np.tile(c[\'history_c\'],len(states)),d=np.tile(c[\'history_d\'],len(states)))\n\ndef history(t,m,c):\n return m[\'s\']+c[\'history_amplitude\']*(m[\'c\']*np.sin(c[\'history_frequency\']*t)+m[\'d\']*(1-np.cos(c[\'history_frequency\']*t)))\n\ndef simulate(m,h,c):\n n=int(round(c[\'horizon\']/h));stride=int(round(c[\'fine_observation_step\']/h));p=c[\'generator\']\n assert np.isclose(stride*h,c[\'fine_observation_step\']) and h<c[\'delay_bounds\'][0]\n y=np.empty((len(m[\'s\']),n+1));y[:,0]=m[\'s\']\n for i in range(n):\n  def rhs(t,x):\n   q=t-p[\'true_delay\']\n   if q<0:past=history(q,m,c)\n   else:\n    left=max(0,int(np.floor(q/h)));w=q/h-left;assert left+1<=i\n    past=(1-w)*y[:,left]+w*y[:,left+1]\n   return p[\'growth\']*x*(1-past/p[\'capacity\'])\n  t=i*h;x=y[:,i];a=rhs(t,x);b=rhs(t+h/2,x+h*a/2);d=rhs(t+h/2,x+h*b/2);e=rhs(t+h,x+h*d)\n  y[:,i+1]=x+h*(a+2*b+2*d+e)/6\n assert np.isfinite(y).all() and (y>0).all()\n return y[:,::stride]\n\ndef reference(c):\n splits=[\'train\',\'validation\',\'test\'];ms={s:metadata(c[s+\'_states\'],c) for s in splits}\n merged={k:np.concatenate([ms[s][k] for s in splits]) for k in (\'s\',\'c\',\'d\')};previous=None;checks=[]\n for h in c[\'reference_steps\']:\n  result=simulate(merged,h,c)\n  if previous is not None:\n   delta=float(np.max(abs(result-previous)));checks.append(dict(coarser_step=old,finer_step=h,max_difference=delta,passes=delta<=c[\'reference_tolerance\']))\n   if checks[-1][\'passes\']:break\n  previous=result;old=h\n else:raise RuntimeError(\'Reference convergence failed\')\n out={};offset=0\n for s in splits:\n  n=len(ms[s][\'s\']);out[s]=dict(metadata=ms[s],fine_clean=result[offset:offset+n]);offset+=n\n return out,dict(selected_step=h,comparisons=checks,tolerance=c[\'reference_tolerance\'],observation_grid_step=c[\'fine_observation_step\'],scope=\'Empirical 40-trajectory convergence, not a rigorous enclosure.\')\n\ndef anchors(noisy,c):\n n=int(round(c[\'slope_window_duration\']/c[\'observation_step\']))+1;t=np.arange(n)*c[\'observation_step\']\n grid=np.linspace(*c[\'delay_bounds\'],c[\'delay_profile_points\']);est=[];slopes=[];profiles=[];norms=[]\n for i,s in enumerate(c[\'train_states\']):\n  coef=np.polyfit(t/t[-1],noisy[i*4:i*4+4,:n].T,deg=c[\'slope_polynomial_degree\']);v=coef[-2]/t[-1];slopes.append(v)\n  q=np.array([history(-r,metadata([s],c),c) for r in grid]);w=q-q.mean(axis=1,keepdims=True);norm=np.linalg.norm(w,axis=1)\n  assert norm.min()>c[\'geometry_norm_threshold\'];v=v-v.mean();res=np.maximum(float(v@v)-(w@v)**2/norm**2,0)/4\n  est.append(grid[int(res.argmin())]);profiles.append(res);norms.append(norm.min())\n return dict(delays=np.array(est),slope_estimates=np.array(slopes),profiles=np.array(profiles),minimum_signature_norm=np.array(norms))\n\ndef observed(ref,c,seed,sigma):\n stride=int(round(c[\'observation_step\']/c[\'fine_observation_step\']));out={}\n for split,offset in [(\'train\',0),(\'validation\',2000),(\'test\',1000)]:\n  b=ref[split];clean=b[\'fine_clean\'];noisy=clean.copy();noisy[:,1:]+=sigma*np.random.default_rng(seed+offset).normal(size=noisy[:,1:].shape)\n  out[split]=dict(metadata=b[\'metadata\'],fine_clean=clean,fine_noisy=noisy,clean=clean[:,::stride],noisy=noisy[:,::stride])\n out[\'anchors\']=anchors(out[\'train\'][\'noisy\'],c);return out\n\ndef save(path,case):\n fields={}\n for s in [\'train\',\'validation\',\'test\']:\n  for k,v in case[s].items():\n   if k==\'metadata\':fields.update({s+\'_meta_\'+m:a for m,a in v.items()})\n   else:fields[s+\'_\'+k]=v\n fields.update({\'anchors_\'+k:v for k,v in case[\'anchors\'].items()});np.savez_compressed(path,**fields)\n\ndef load(path):\n with np.load(path) as f:\n  out={s:{k:f[s+\'_\'+k] for k in [\'clean\',\'noisy\',\'fine_clean\',\'fine_noisy\']} for s in [\'train\',\'validation\',\'test\']}\n  for s in out:out[s][\'metadata\']={k:f[s+\'_meta_\'+k] for k in [\'s\',\'c\',\'d\']}\n  out[\'anchors\']={k[8:]:f[k] for k in f.files if k.startswith(\'anchors_\')}\n return out\n', 'e4_neural.py': '"""Generic PyTorch architectures; pinned external Euler/direct solver only."""\nimport hashlib,random,time\nimport numpy as np\nimport torch\nfrom torch import nn\nfrom torchdde import integrate,Euler,ConstantStepSizeController\ntorch.set_num_threads(1)\n\ndef seed(s):random.seed(s);np.random.seed(s);torch.manual_seed(s)\n\ndef mlp(w):return nn.Sequential(nn.Linear(1,w),nn.Tanh(),nn.Linear(w,w),nn.Tanh(),nn.Linear(w,1))\n\nclass Model(nn.Module):\n def __init__(self,method,initial):\n  super().__init__();self.delays=nn.Parameter(torch.tensor([initial],dtype=torch.float32));self.external=method.startswith(\'External\')\n  if self.external:\n   dims=[2,32,32,32,32,1];layers=[]\n   for i,(a,b) in enumerate(zip(dims,dims[1:])):\n    layers.append(nn.Linear(a,b))\n    if i<len(dims)-2:layers.append(nn.ReLU())\n   self.field=nn.Sequential(*layers)\n  else:self.a_net=mlp(28);self.b_net=mlp(22)\n def forward(self,t,x,args,*,history):\n  if self.external:return self.field(torch.cat([x,*history],dim=-1))\n  return self.a_net(x)-(0.05+torch.nn.functional.softplus(self.b_net(x)))*history[0]\n\ndef state_hash(model):\n h=hashlib.sha256()\n for k,v in model.state_dict().items():h.update(k.encode());h.update(v.detach().numpy().tobytes())\n return h.hexdigest()\n\ndef rollout(model,meta,c,obs,h,T=None):\n T=c[\'horizon\'] if T is None else T;ts=torch.arange(int(round(T/obs))+1,dtype=torch.float32)*obs\n m={k:torch.tensor(v,dtype=torch.float32).reshape(-1,1) for k,v in meta.items()}\n def history(t):return m[\'s\']+c[\'history_amplitude\']*(m[\'c\']*torch.sin(c[\'history_frequency\']*t)+m[\'d\']*(1-torch.cos(c[\'history_frequency\']*t)))\n y=integrate(model,Euler(),ts[0],ts[-1],ts,history,None,stepsize_controller=ConstantStepSizeController(),dt0=torch.tensor(h),delays=model.delays,discretize_then_optimize=True,max_steps=2048)\n assert y.shape==(len(meta[\'s\']),len(ts),1) and torch.equal(y[:,0],m[\'s\']);return y[:,:,0]\n\ndef fit(r,case,c):\n seed(r[\'model_seed\']);m=Model(r[\'method\'],c[\'initial_delay\']);initial=state_hash(m);pid=id(m.delays);ps=list(m.parameters())\n opt=torch.optim.Adam([{\'params\':[p for n,p in m.named_parameters() if n!=\'delays\'],\'weight_decay\':c[\'field_weight_decay\']},{\'params\':[m.delays],\'weight_decay\':0.}],lr=r[\'learning_rate\'])\n target=torch.tensor(case[\'train\'][\'noisy\'],dtype=torch.float32);anchors=torch.tensor(case[\'anchors\'][\'delays\'],dtype=torch.float32)\n updates=clipped=projected=boundary=0;curve=[];start=time.perf_counter()\n for T,count in c[\'schedules\'][str(r[\'budget\'])]:\n  n=int(round(T/c[\'observation_step\']));weights=torch.linspace(.6,1.4,n+1).reshape(1,-1)\n  for _ in range(count):\n   opt.zero_grad(set_to_none=True);y=rollout(m,case[\'train\'][\'metadata\'],c,c[\'observation_step\'],c[\'training_step\'],T)\n   loss=torch.mean(weights*(y-target[:,:n+1])**2)\n   if r[\'method\'].endswith(\'+anchors\'):loss=loss+c[\'anchor_weight\']*torch.mean((m.delays[0]-anchors)**2)\n   if not torch.isfinite(loss):raise FloatingPointError(\'Nonfinite loss\')\n   loss.backward()\n   if any(p.grad is None or not torch.isfinite(p.grad).all() for p in ps):raise FloatingPointError(\'Invalid gradient\')\n   norm=torch.nn.utils.clip_grad_norm_(ps,c[\'gradient_clip\'],error_if_nonfinite=True);clipped+=int(float(norm)>c[\'gradient_clip\']);opt.step()\n   with torch.no_grad():\n    before=float(m.delays[0]);m.delays.clamp_(*c[\'delay_bounds\']);projected+=int(before!=float(m.delays[0]));boundary+=int(float(m.delays[0])<=.1500001 or float(m.delays[0])>=.4999999)\n   assert id(m.delays)==pid;updates+=1\n   if updates==1 or updates%20==0 or updates==r[\'budget\']:curve.append(dict(update=updates,loss_before_update=float(loss.detach()),delay_after_update=float(m.delays.detach()[0])))\n assert updates==r[\'budget\'] and all(torch.isfinite(p).all() for p in ps)\n with torch.no_grad():\n  y=rollout(m,case[\'train\'][\'metadata\'],c,c[\'observation_step\'],c[\'training_step\']);tr=float(torch.mean(torch.linspace(.6,1.4,y.shape[1]).reshape(1,-1)*(y-target)**2));an=float(torch.mean((m.delays[0]-anchors)**2))\n return m,dict(initial_parameters_hash=initial,final_parameters_hash=state_hash(m),parameter_count=sum(p.numel() for p in ps),updates=updates,clipped_updates=clipped,projected_updates=projected,boundary_updates=boundary,training_seconds=time.perf_counter()-start,final_weighted_training_mse=tr,final_anchor_fit_mse=an,final_objective=tr+(c[\'anchor_weight\']*an if r[\'method\'].endswith(\'+anchors\') else 0),optimizer_parameter_identity_preserved=True),curve\n\ndef evaluate(m,case,c,stage):\n est=float(m.delays.detach()[0]);out=dict(estimated_delay=est,absolute_delay_error=abs(est-c[\'generator\'][\'true_delay\']))\n specs=[(c[\'training_step\'],c[\'observation_step\'],[\'train\',\'validation\'])] if stage==\'calibration\' else [(h,c[\'fine_observation_step\'],[\'train\',\'test\']) for h in c[\'evaluation_steps\']]\n with torch.no_grad():\n  for h,obs,splits in specs:\n   preds=[]\n   for s in splits:\n    y=rollout(m,case[s][\'metadata\'],c,obs,h).numpy().astype(float)\n    if not np.isfinite(y).all():raise FloatingPointError(\'Nonfinite prediction\')\n    preds.append(y);prefix=\'\' if stage==\'calibration\' else \'fine_\'\n    for name in [\'clean\',\'noisy\']:out[f\'h{h:g}_{s}_{name}_mse\']=float(np.mean((y-case[s][prefix+name])**2))\n   out[f\'h{h:g}_minimum_observation\']=float(min(y.min() for y in preds));out[f\'h{h:g}_nonpositive_observations\']=int(sum((y<=0).sum() for y in preds))\n return out\n', 'e4_analysis.py': '"""Fixed selection rule and descriptive paired summaries; no outcome tuning."""\nimport itertools,json\nfrom pathlib import Path\nimport numpy as np\n\ndef write(path,value):\n path=Path(path);tmp=path.with_suffix(path.suffix+\'.tmp\');tmp.write_text(json.dumps(value,indent=2,allow_nan=False)+\'\\n\');tmp.replace(path)\n\ndef cell(m,s):return f\'{m}|sigma{s:g}\'\n\ndef choose(rows,c):\n out={}\n for m,s in itertools.product(c[\'methods\'],c[\'noise_levels\']):\n  trials=[]\n  for lr,b in itertools.product(c[\'learning_rates\'],map(int,c[\'schedules\'])):\n   g=[r for r in rows if (r[\'method\'],r[\'noise_std\'],r[\'learning_rate\'],r[\'budget\'])==(m,s,lr,b)]\n   eligible=len(g)==len(c[\'calibration_data_seeds\']) and {r[\'data_seed\'] for r in g}==set(c[\'calibration_data_seeds\']) and all(r[\'status\']==\'ok\' for r in g)\n   trials.append(dict(learning_rate=lr,budget=b,eligible=eligible,score=float(np.mean([r[\'h0.025_validation_noisy_mse\'] for r in g])) if eligible else None))\n  eligible=[t for t in trials if t[\'eligible\']];best=min(eligible,key=lambda t:(t[\'score\'],t[\'budget\'],t[\'learning_rate\'])) if eligible else None\n  out[cell(m,s)]=dict(selected=best,trials=trials,available=best is not None)\n return out\n\ndef summarize(rows,direct,c,root):\n import pandas as pd\n outcomes=[\'absolute_delay_error\',\'h0.00625_test_clean_mse\'];blocks=[];medians=[];paired=[]\n for m,s,seed in itertools.product(c[\'methods\'],c[\'noise_levels\'],c[\'final_data_seeds\']):\n  g=[r for r in rows if (r[\'method\'],r[\'noise_std\'],r[\'data_seed\'])==(m,s,seed)]\n  if len(g)==len(c[\'optimization_seeds\']) and {r[\'optimization_seed\'] for r in g}==set(c[\'optimization_seeds\']) and all(r[\'status\']==\'ok\' for r in g):\n   blocks.append(dict(method=m,noise_std=s,data_seed=seed,**{k:float(np.median([r[k] for r in g])) for k in outcomes}))\n for m,s in itertools.product(c[\'methods\'],c[\'noise_levels\']):\n  g=[b for b in blocks if (b[\'method\'],b[\'noise_std\'])==(m,s)]\n  if len(g)==len(c[\'final_data_seeds\']):medians.append(dict(method=m,noise_std=s,**{k:float(np.median([b[k] for b in g])) for k in outcomes},**{k+\'_range\':[min(b[k] for b in g),max(b[k] for b in g)] for k in outcomes}))\n pairs=[(\'External-C+anchors\',\'External-C\'),(\'Structured-C+anchors\',\'Structured-C\'),(\'Structured-C\',\'External-C\'),(\'Structured-C+anchors\',\'External-C+anchors\')]\n for (a,b),s,k in itertools.product(pairs,c[\'noise_levels\'],outcomes):\n  av={r[\'data_seed\']:r[k] for r in blocks if (r[\'method\'],r[\'noise_std\'])==(a,s)};bv={r[\'data_seed\']:r[k] for r in blocks if (r[\'method\'],r[\'noise_std\'])==(b,s)}\n  complete=set(av)==set(bv)==set(c[\'final_data_seeds\']);vals=[dict(data_seed=d,numerator=av[d],denominator=bv[d],ratio=av[d]/bv[d] if bv[d]!=0 else None,difference=av[d]-bv[d]) for d in c[\'final_data_seeds\']] if complete else []\n  ratios=[v[\'ratio\'] for v in vals if v[\'ratio\'] is not None]\n  paired.append(dict(numerator_method=a,denominator_method=b,noise_std=s,outcome=k,complete=complete,datasets=vals,median_ratio=float(np.median(ratios)) if complete and len(ratios)==len(vals) else None,ratio_range=[min(ratios),max(ratios)] if ratios else None,median_difference=float(np.median([v[\'difference\'] for v in vals])) if complete else None,dataset_wins=sum(v[\'difference\']<0 for v in vals) if complete else None))\n result=dict(expected_final_fits=len(c[\'methods\'])*len(c[\'noise_levels\'])*len(c[\'final_data_seeds\'])*len(c[\'optimization_seeds\']),recorded_final_fits=len(rows),successful_final_fits=sum(r[\'status\']==\'ok\' for r in rows),failed_final_fits=sum(r[\'status\']!=\'ok\' for r in rows),direct_estimates=direct,dataset_medians=blocks,method_medians=medians,paired_comparisons=paired,independent_base_noise_datasets=len(c[\'final_data_seeds\']),interpretation=\'Descriptive paired results. Optimizer repeats and noise levels are not independent datasets; no significance claim.\')\n write(root/\'summary.json\',result);pd.DataFrame(rows).to_csv(root/\'final_runs.csv\',index=False);pd.DataFrame(blocks).to_csv(root/\'dataset_medians.csv\',index=False);pd.DataFrame(direct).to_csv(root/\'direct_anchors.csv\',index=False)\n return result\n', 'run.py': '"""Restartable fixed E4 recovery run with separately saved stage boundaries."""\nimport argparse,hashlib,itertools,json,multiprocessing,platform,subprocess,time,traceback\nfrom concurrent.futures import ProcessPoolExecutor,as_completed\nfrom datetime import datetime,timezone\nfrom pathlib import Path\nimport numpy as np\nimport torch,torchdde\nimport e4_data as data\nimport e4_neural as neural\nfrom e4_analysis import write,choose,cell,summarize\n\nHERE=Path(__file__).resolve().parent\nFROZEN=[\'protocol.json\',\'PROTOCOL.md\',\'e4_data.py\',\'e4_neural.py\',\'e4_analysis.py\',\'run.py\',\'audit.py\',\'validate.py\']\ndef utc():return datetime.now(timezone.utc).isoformat()\ndef sha(p):return hashlib.sha256(Path(p).read_bytes()).hexdigest()\ndef case_name(s,d):return f\'sigma{s:g}_data{d}.npz\'\ndef stem(r):return f"{r[\'stage\']}_sigma{r[\'noise_std\']:g}_data{r[\'data_seed\']}_{r[\'method\']}_lr{r[\'learning_rate\']:g}_u{r[\'budget\']}_seed{r[\'model_seed\']}"\n\ndef source(root,commit):\n root=Path(root).resolve();actual=subprocess.check_output([\'git\',\'-C\',str(root),\'rev-parse\',\'HEAD\'],text=True).strip();assert actual==commit\n assert not subprocess.check_output([\'git\',\'-C\',str(root),\'diff\',\'HEAD\',\'--name-only\'],text=True).strip()\n names=subprocess.check_output([\'git\',\'-C\',str(root),\'ls-files\',\'-z\']).decode().split(\'\\0\')\n return dict(commit=commit,tracked_sha256={n:sha(root/n) for n in names if n})\n\ndef signature(c,solver,architecture):\n assert Path(torchdde.__file__).resolve().parent.parent==solver.resolve()\n return dict(config=c,source_sha256={n:sha(HERE/n) for n in FROZEN},solver=source(solver,c[\'solver_commit\']),architecture=source(architecture,c[\'architecture_source_commit\']),runtime=dict(python=platform.python_version(),torch=torch.__version__,numpy=np.__version__,workers=c[\'workers\'],threads_per_worker=1,device=\'cpu\'))\n\ndef worker(r,c,root):\n start=time.perf_counter();r=dict(r)\n try:\n  case=data.load(root/\'observations\'/case_name(r[\'noise_std\'],r[\'data_seed\']));m,fit,curve=neural.fit(r,case,c);r.update(fit)\n  path=root/\'checkpoints\'/f\'{stem(r)}.pt\';temp=path.with_suffix(\'.tmp\');torch.save(m.state_dict(),temp);temp.replace(path)\n  write(root/\'curves\'/f\'{stem(r)}.json\',curve);r.update(neural.evaluate(m,case,c,r[\'stage\']),status=\'ok\')\n except Exception as e:\n  r.update(status=\'failed\',error=f\'{type(e).__name__}: {e}\');(root/\'failures\'/f\'{stem(r)}.txt\').write_text(traceback.format_exc())\n r[\'total_job_seconds\']=time.perf_counter()-start;write(root/\'individual_records\'/f\'{stem(r)}.json\',r);return r\n\ndef execute(specs,c,root,stage,limit):\n record=root/f\'{stage}_records.json\';rows=json.loads(record.read_text()) if record.exists() else []\n done={stem(r) for r in rows};expected={stem(r) for r in specs};assert len(done)==len(rows) and done.issubset(expected)\n # Recover an atomic worker record if its parent was interrupted before commit.\n for p in (root/\'individual_records\').glob(stage+\'_*.json\'):\n  r=json.loads(p.read_text())\n  if stem(r) not in done:assert stem(r) in expected;rows.append(r);done.add(stem(r))\n jobs=[r for r in specs if stem(r) not in done]\n if limit:jobs=jobs[:limit]\n with ProcessPoolExecutor(max_workers=c[\'workers\'],mp_context=multiprocessing.get_context(\'spawn\')) as pool:\n  futures=[]\n  for r in jobs:\n   with (root/\'events.jsonl\').open(\'a\') as f:f.write(json.dumps(dict(event=\'submit_or_restart\',utc=utc(),run=stem(r)))+\'\\n\')\n   futures.append(pool.submit(worker,r,c,root))\n  for f in as_completed(futures):\n   r=f.result();rows.append(r);rows.sort(key=stem);write(record,rows);print(f"{stage} {len(rows)}/{len(specs)} {stem(r)} {r[\'status\']} seconds={r[\'total_job_seconds\']:.1f}",flush=True)\n rows.sort(key=stem);write(record,rows);return rows,len(rows)==len(specs)\n\ndef main():\n p=argparse.ArgumentParser();p.add_argument(\'--results\',type=Path,required=True);p.add_argument(\'--solver\',type=Path,required=True);p.add_argument(\'--architecture\',type=Path,required=True);p.add_argument(\'--stage\',choices=[\'calibration\',\'final\'],required=True);p.add_argument(\'--mode\',choices=[\'smoke\',\'full\'],default=\'full\');p.add_argument(\'--limit\',type=int,default=0);a=p.parse_args()\n c=json.loads((HERE/\'protocol.json\').read_text())\n if a.mode==\'smoke\':c.update(c[\'smoke\'])\n c[\'mode\']=a.mode;root=a.results.resolve();root.mkdir(parents=True,exist_ok=True);sig=signature(c,a.solver,a.architecture)\n path=root/\'manifest.json\'\n if path.exists():assert json.loads(path.read_text())[\'signature\']==sig,\'Source/settings/runtime changed\'\n else:\n  assert not any(root.iterdir());write(path,dict(frozen_before_this_recovery_run_utc=utc(),signature=sig,recovery_disclosure=\'Original attempt outputs were lost during workspace maintenance. Its calibration completion and selected configurations were seen. This is a reconstructed rerun under the same scientific settings, with eight rather than four one-thread workers; not a fresh public preregistration.\'))\n for folder in [\'observations\',\'checkpoints\',\'curves\',\'failures\',\'individual_records\']:(root/folder).mkdir(exist_ok=True)\n ref,convergence=data.reference(c);write(root/\'reference_convergence.json\',convergence)\n for s,d in itertools.product(c[\'noise_levels\'],c[\'calibration_data_seeds\']+c[\'final_data_seeds\']):\n  path=root/\'observations\'/case_name(s,d)\n  if not path.exists():data.save(path,data.observed(ref,c,d,s))\n if a.stage==\'calibration\':\n  assert not (root/\'final_records.json\').exists(),\'Calibration cannot be altered after final fitting\'\n  specs=[dict(stage=\'calibration\',method=m,noise_std=s,learning_rate=lr,budget=b,data_seed=d,model_seed=c[\'calibration_model_seed\']+100*i) for m,s,lr,b,(i,d) in itertools.product(c[\'methods\'],c[\'noise_levels\'],c[\'learning_rates\'],map(int,c[\'schedules\']),enumerate(c[\'calibration_data_seeds\']))]\n  rows,complete=execute(specs,c,root,\'calibration\',a.limit)\n  if complete:\n   lock=dict(locked_utc=utc(),calibration_records_sha256=sha(root/\'calibration_records.json\'),selections=choose(rows,c))\n   path=root/\'selection_lock.json\'\n   if path.exists():assert json.loads(path.read_text())[\'selections\']==lock[\'selections\']\n   else:write(path,lock)\n   print(\'CALIBRATION LOCKED\',flush=True)\n else:\n  lock=json.loads((root/\'selection_lock.json\').read_text());assert lock[\'calibration_records_sha256\']==sha(root/\'calibration_records.json\')\n  specs=[];direct=[]\n  for s,(i,d) in itertools.product(c[\'noise_levels\'],enumerate(c[\'final_data_seeds\'])):\n   case=data.load(root/\'observations\'/case_name(s,d));est=float(np.clip(case[\'anchors\'][\'delays\'].mean(),*c[\'delay_bounds\']))\n   direct.append(dict(method=\'Anchor-C\',noise_std=s,data_seed=d,estimated_delay=est,absolute_delay_error=abs(est-c[\'generator\'][\'true_delay\'])))\n   for m,opt in itertools.product(c[\'methods\'],c[\'optimization_seeds\']):\n    selected=lock[\'selections\'][cell(m,s)][\'selected\']\n    if selected is not None:specs.append(dict(stage=\'final\',method=m,noise_std=s,data_seed=d,optimization_seed=opt,model_seed=opt+100*i,learning_rate=selected[\'learning_rate\'],budget=selected[\'budget\']))\n  write(root/\'direct_records.json\',direct);rows,complete=execute(specs,c,root,\'final\',a.limit)\n  if complete:\n   summarize(rows,direct,c,root);write(root/\'execution_complete.json\',dict(completed_utc=utc(),final_records=len(rows),successful=sum(r[\'status\']==\'ok\' for r in rows)));print(\'FINAL COMPLETE\',flush=True)\n\nif __name__==\'__main__\':main()\n', 'audit.py': '"""Consistency replay; shared model/solver, independent score/selection arithmetic."""\nimport argparse,itertools,json\nfrom pathlib import Path\nimport numpy as np\nimport torch,torchdde\nimport e4_data as data\nimport e4_neural as neural\nfrom e4_analysis import write\nfrom run import HERE,FROZEN,sha,source,stem,case_name\n\ndef audit(root,architecture):\n manifest=json.loads((root/\'manifest.json\').read_text());sig=manifest[\'signature\'];c=sig[\'config\'];checks=[]\n def check(name,value):\n  checks.append(dict(check=name,passed=bool(value)))\n  if not value:raise AssertionError(name)\n for n in FROZEN:check(\'frozen \'+n,sha(HERE/n)==sig[\'source_sha256\'][n])\n solver=Path(torchdde.__file__).resolve().parent.parent\n check(\'solver provenance\',source(solver,c[\'solver_commit\'])==sig[\'solver\']);check(\'architecture provenance\',source(architecture,c[\'architecture_source_commit\'])==sig[\'architecture\'])\n check(\'runtime torch\',torch.__version__==sig[\'runtime\'][\'torch\']);check(\'runtime numpy\',np.__version__==sig[\'runtime\'][\'numpy\'])\n ref,conv=data.reference(c);check(\'reference\',conv==json.loads((root/\'reference_convergence.json\').read_text()));cases={}\n for s,d in itertools.product(c[\'noise_levels\'],c[\'calibration_data_seeds\']+c[\'final_data_seeds\']):\n  expected=data.observed(ref,c,d,s);actual=data.load(root/\'observations\'/case_name(s,d));cases[(s,d)]=actual\n  for split in [\'train\',\'validation\',\'test\']:\n   for k in [\'clean\',\'noisy\',\'fine_clean\',\'fine_noisy\']:check(f\'data/{s}/{d}/{split}/{k}\',np.array_equal(expected[split][k],actual[split][k]))\n   for k in [\'s\',\'c\',\'d\']:check(f\'meta/{s}/{d}/{split}/{k}\',np.array_equal(expected[split][\'metadata\'][k],actual[split][\'metadata\'][k]))\n  for k,v in expected[\'anchors\'].items():check(f\'anchors/{s}/{d}/{k}\',np.array_equal(v,actual[\'anchors\'][k]))\n cal=json.loads((root/\'calibration_records.json\').read_text());final=json.loads((root/\'final_records.json\').read_text());lock=json.loads((root/\'selection_lock.json\').read_text())\n check(\'calibration complete\',len(cal)==len(c[\'methods\'])*len(c[\'noise_levels\'])*len(c[\'learning_rates\'])*len(c[\'schedules\'])*len(c[\'calibration_data_seeds\']))\n check(\'calibration hash\',sha(root/\'calibration_records.json\')==lock[\'calibration_records_sha256\'])\n check(\'unique records\',len({stem(r) for r in cal+final})==len(cal)+len(final));independent={}\n for m,s in itertools.product(c[\'methods\'],c[\'noise_levels\']):\n  trials=[]\n  for lr,b in itertools.product(c[\'learning_rates\'],map(int,c[\'schedules\'])):\n   g=[r for r in cal if (r[\'method\'],r[\'noise_std\'],r[\'learning_rate\'],r[\'budget\'])==(m,s,lr,b)]\n   ok=len(g)==len(c[\'calibration_data_seeds\']) and {r[\'data_seed\'] for r in g}==set(c[\'calibration_data_seeds\']) and all(r[\'status\']==\'ok\' for r in g)\n   trials.append(dict(learning_rate=lr,budget=b,eligible=ok,score=sum(r[\'h0.025_validation_noisy_mse\'] for r in g)/len(g) if ok else None))\n  eligible=[t for t in trials if t[\'eligible\']];best=sorted(eligible,key=lambda t:(t[\'score\'],t[\'budget\'],t[\'learning_rate\']))[0] if eligible else None\n  independent[f\'{m}|sigma{s:g}\']=dict(selected=best,trials=trials,available=best is not None)\n check(\'independent selection arithmetic\',independent==lock[\'selections\'])\n expected=[]\n for m,s,(i,d),opt in itertools.product(c[\'methods\'],c[\'noise_levels\'],enumerate(c[\'final_data_seeds\']),c[\'optimization_seeds\']):\n  if independent[f\'{m}|sigma{s:g}\'][\'available\']:expected.append((m,s,d,opt,opt+100*i))\n check(\'expected final cells\',sorted(expected)==sorted((r[\'method\'],r[\'noise_std\'],r[\'data_seed\'],r[\'optimization_seed\'],r[\'model_seed\']) for r in final))\n events=[json.loads(t) for t in (root/\'events.jsonl\').read_text().splitlines()];check(\'lock before final submissions\',all(e[\'utc\']>=lock[\'locked_utc\'] for e in events if e[\'run\'].startswith(\'final_\')))\n maxdiff=0.;replayed=0;initgroups={}\n for r in cal+final:\n  if r[\'status\']!=\'ok\':check(\'failure retained \'+stem(r),(root/\'failures\'/f\'{stem(r)}.txt\').exists());continue\n  neural.seed(r[\'model_seed\']);m=neural.Model(r[\'method\'],c[\'initial_delay\']);check(\'initial hash \'+stem(r),neural.state_hash(m)==r[\'initial_parameters_hash\'])\n  initgroups.setdefault((r[\'method\'].split(\'+\')[0],r[\'model_seed\']),set()).add(r[\'initial_parameters_hash\'])\n  m.load_state_dict(torch.load(root/\'checkpoints\'/f\'{stem(r)}.pt\',map_location=\'cpu\',weights_only=True));check(\'checkpoint hash \'+stem(r),neural.state_hash(m)==r[\'final_parameters_hash\'])\n  check(\'finite weights\',all(torch.isfinite(p).all() for p in m.parameters()));check(\'updates\',r[\'updates\']==r[\'budget\']==sum(x[1] for x in c[\'schedules\'][str(r[\'budget\'])]))\n  if r[\'stage\']==\'final\':\n   selected=independent[f"{r[\'method\']}|sigma{r[\'noise_std\']:g}"][\'selected\'];check(\'selected final settings\',(r[\'learning_rate\'],r[\'budget\'])==(selected[\'learning_rate\'],selected[\'budget\']))\n  case=cases[(r[\'noise_std\'],r[\'data_seed\'])];est=float(m.delays.detach()[0]);values=dict(estimated_delay=est,absolute_delay_error=abs(est-c[\'generator\'][\'true_delay\']))\n  specs=[(c[\'training_step\'],c[\'observation_step\'],[\'train\',\'validation\'])] if r[\'stage\']==\'calibration\' else [(h,c[\'fine_observation_step\'],[\'train\',\'test\']) for h in c[\'evaluation_steps\']]\n  with torch.no_grad():\n   for h,obs,splits in specs:\n    predictions=[]\n    for split in splits:\n     y=neural.rollout(m,case[split][\'metadata\'],c,obs,h).numpy().astype(float);predictions.append(y);prefix=\'\' if r[\'stage\']==\'calibration\' else \'fine_\'\n     for name in [\'clean\',\'noisy\']:\n      residual=y-case[split][prefix+name];values[f\'h{h:g}_{split}_{name}_mse\']=float(np.sum(residual**2)/residual.size)\n    values[f\'h{h:g}_minimum_observation\']=float(min(y.min() for y in predictions));values[f\'h{h:g}_nonpositive_observations\']=int(sum((y<=0).sum() for y in predictions))\n   pred=neural.rollout(m,case[\'train\'][\'metadata\'],c,c[\'observation_step\'],c[\'training_step\']);target=torch.tensor(case[\'train\'][\'noisy\'],dtype=torch.float32)\n   tr=float(torch.mean(torch.linspace(.6,1.4,pred.shape[1]).reshape(1,-1)*(pred-target)**2));an=float(torch.mean((m.delays[0]-torch.tensor(case[\'anchors\'][\'delays\'],dtype=torch.float32))**2))\n   values.update(final_weighted_training_mse=tr,final_anchor_fit_mse=an,final_objective=tr+(c[\'anchor_weight\']*an if r[\'method\'].endswith(\'+anchors\') else 0))\n  for k,v in values.items():maxdiff=max(maxdiff,abs(v-r[k]));check(\'metric \'+stem(r)+\'/\'+k,np.isclose(v,r[k],rtol=1e-9,atol=1e-12))\n  replayed+=1\n check(\'paired initialization\',all(len(v)==1 for v in initgroups.values()))\n direct=json.loads((root/\'direct_records.json\').read_text())\n for r in direct:\n  est=float(np.clip(cases[(r[\'noise_std\'],r[\'data_seed\'])][\'anchors\'][\'delays\'].mean(),*c[\'delay_bounds\']));check(\'direct \'+str((r[\'noise_std\'],r[\'data_seed\'])),est==r[\'estimated_delay\'] and abs(est-c[\'generator\'][\'true_delay\'])==r[\'absolute_delay_error\'])\n summary=json.loads((root/\'summary.json\').read_text());blocks=summary[\'dataset_medians\'];outcomes=[\'absolute_delay_error\',\'h0.00625_test_clean_mse\']\n for b in blocks:\n  g=[r for r in final if (r[\'method\'],r[\'noise_std\'],r[\'data_seed\'])==(b[\'method\'],b[\'noise_std\'],b[\'data_seed\']) and r[\'status\']==\'ok\'];check(\'three repeats\',len(g)==len(c[\'optimization_seeds\']))\n  for k in outcomes:check(\'dataset median \'+k,float(np.median([r[k] for r in g]))==b[k])\n for p in summary[\'paired_comparisons\']:\n  if not p[\'complete\']:continue\n  ratios=[];differences=[]\n  for d in p[\'datasets\']:\n   a=next(b[p[\'outcome\']] for b in blocks if (b[\'method\'],b[\'noise_std\'],b[\'data_seed\'])==(p[\'numerator_method\'],p[\'noise_std\'],d[\'data_seed\']));b=next(b[p[\'outcome\']] for b in blocks if (b[\'method\'],b[\'noise_std\'],b[\'data_seed\'])==(p[\'denominator_method\'],p[\'noise_std\'],d[\'data_seed\']))\n   check(\'paired values\',a==d[\'numerator\'] and b==d[\'denominator\']);differences.append(a-b)\n   if b!=0:ratios.append(a/b)\n  check(\'paired difference\',float(np.median(differences))==p[\'median_difference\']);check(\'paired wins\',sum(v<0 for v in differences)==p[\'dataset_wins\'])\n  if len(ratios)==len(p[\'datasets\']):check(\'paired ratio\',float(np.median(ratios))==p[\'median_ratio\'])\n for m in summary[\'method_medians\']:\n  g=[b for b in blocks if (b[\'method\'],b[\'noise_std\'])==(m[\'method\'],m[\'noise_std\'])];check(\'five datasets\',len(g)==len(c[\'final_data_seeds\']))\n  for k in outcomes:check(\'marginal median\',float(np.median([b[k] for b in g]))==m[k]);check(\'marginal range\',[min(b[k] for b in g),max(b[k] for b in g)]==m[k+\'_range\'])\n report=dict(status=\'passed\',checkpoints_replayed=replayed,calibration_records=len(cal),final_records=len(final),observation_datasets=len(cases),direct_estimates=len(direct),checks=len(checks),maximum_metric_difference=maxdiff,scope=\'Consistency replay with independent score/selection arithmetic; shared architecture/solver, not independent research replication.\')\n write(root/\'audit_checks.json\',checks);write(root/\'audit_report.json\',report);return report\n\nif __name__==\'__main__\':\n p=argparse.ArgumentParser();p.add_argument(\'--results\',type=Path,required=True);p.add_argument(\'--architecture\',type=Path,required=True);a=p.parse_args();print(json.dumps(audit(a.results.resolve(),a.architecture.resolve()),indent=2))\n', 'validate.py': '"""Reserved component checks, including direct-mode finite differences."""\nimport copy,json,math\nfrom pathlib import Path\nimport numpy as np\nimport torch\nfrom torch import nn\nfrom torchdde import integrate,Euler,ConstantStepSizeController\nimport e4_neural as neural\nimport e4_data as data\nfrom e4_analysis import choose,cell\n\ndef gradient_fixture(feedback):\n class Field(nn.Module):\n  def __init__(self):\n   super().__init__();self.delays=nn.Parameter(torch.tensor([.313],dtype=torch.float64));self.beta=nn.Parameter(torch.tensor(.12 if feedback else .1,dtype=torch.float64));self.alpha=nn.Parameter(torch.tensor(-.45 if feedback else .4*math.exp(.5*.313),dtype=torch.float64))\n  def forward(self,t,x,args,*,history):return self.beta*x+self.alpha*history[0]\n m=Field();T=.8 if feedback else .2;h=.0125 if feedback else .005\n s=torch.tensor([.45,.70,.95,1.2],dtype=torch.float64).reshape(-1,1);cv=torch.tensor([-1,0,1,0],dtype=torch.float64).reshape(-1,1);dv=torch.tensor([0,1,0,-1],dtype=torch.float64).reshape(-1,1)\n def hist(t):return s+.18*(cv*torch.sin(math.pi*t)+dv*(1-torch.cos(math.pi*t))) if feedback else torch.exp(.5*t).reshape(1,1)\n ts=torch.linspace(0,T,int(round(T/h))+1,dtype=torch.float64)\n def loss():return integrate(m,Euler(),ts[0],ts[-1],ts,hist,None,stepsize_controller=ConstantStepSizeController(),dt0=torch.tensor(h,dtype=torch.float64),delays=m.delays,discretize_then_optimize=True,max_steps=2048)[:,-1].square().mean()\n value=loss();value.backward();records=[]\n for name,p in m.named_parameters():\n  analytic=float(p.grad.reshape(-1)[0]);delta=1e-6;original=p.detach().clone()\n  with torch.no_grad():p.copy_(original+delta);plus=float(loss());p.copy_(original-delta);minus=float(loss());p.copy_(original)\n  fd=(plus-minus)/(2*delta);relative=abs(analytic-fd)/max(abs(fd),1e-10);assert relative<1e-5,(name,analytic,fd)\n  records.append(dict(parameter=name,returned_gradient=analytic,finite_difference=fd,relative_error=relative))\n return dict(feedback=feedback,records=records)\n\ndef main():\n c=json.loads((Path(__file__).parent/\'protocol.json\').read_text());rows=[]\n for d in c[\'calibration_data_seeds\']:\n  for lr in c[\'learning_rates\']:\n   for b in map(int,c[\'schedules\']):rows.append(dict(method=\'External-C\',noise_std=.0005,learning_rate=lr,budget=b,data_seed=d,status=\'ok\',**{\'h0.025_validation_noisy_mse\':1.}))\n selected=choose(rows,c)[cell(\'External-C\',.0005)][\'selected\'];assert (selected[\'budget\'],selected[\'learning_rate\'])==(280,.0005)\n failed=copy.deepcopy(rows)\n for r in failed:\n  if (r[\'learning_rate\'],r[\'budget\'])==(.0005,280):r[\'status\']=\'failed\'\n assert choose(failed,c)[cell(\'External-C\',.0005)][\'selected\'][\'learning_rate\']==.002\n assert not choose(rows[:1],c)[cell(\'External-C\',.0005)][\'available\']\n assert not choose([],c)[cell(\'External-C\',.0005)][\'available\']\n for name,count in [(\'External-C\',3298),(\'Structured-C\',1471)]:\n  neural.seed(930002);a=neural.Model(name,.325);neural.seed(930002);b=neural.Model(name+\'+anchors\',.325)\n  assert neural.state_hash(a)==neural.state_hash(b) and sum(p.numel() for p in a.parameters())==count\n  y=neural.rollout(a,data.metadata(c[\'train_states\'],c),c,.05,.025,.5);y.square().mean().backward();assert a.delays.grad.abs().max()>0\n  pid=id(a.delays);opt=torch.optim.Adam([a.delays],lr=1.);opt.step()\n  with torch.no_grad():a.delays.clamp_(*c[\'delay_bounds\'])\n  assert id(a.delays)==pid==id(opt.param_groups[0][\'params\'][0])\n ref,conv=data.reference(c);lo=data.observed(ref,c,930001,.0005);hi=data.observed(ref,c,930001,.0015)\n for s in [\'train\',\'validation\',\'test\']:\n  assert np.allclose(3*(lo[s][\'noisy\']-lo[s][\'clean\']),hi[s][\'noisy\']-hi[s][\'clean\'],atol=1e-15,rtol=0)\n  assert np.array_equal(lo[s][\'noisy\'][:,0],lo[s][\'metadata\'][\'s\'])\n print(json.dumps(dict(status=\'passed\',direct_gradient_fixtures=[gradient_fixture(False),gradient_fixture(True)],reference=conv,other_checks=[\'selection tie/missing/failure\',\'paired initialization\',\'parameter counts\',\'delay gradient\',\'projection identity\',\'paired noise/exact initials\']),indent=2))\n\nif __name__==\'__main__\':main()\n', 'README.md': "# ASOC Experiment 4: recovery package\n\nRead REPORT.md for the current completion status and, once available, audited outcomes. This package is updated at stage boundaries and contains the actual raw records/checkpoints available at that save. It is not a revised manuscript or a submission-ready Overleaf project.\n\nThe original in-session attempt was lost during workspace maintenance. Its calibration completion and selected settings had been observed, but its deleted individual records cannot serve as verified results. This reconstructed run repeats every calibration/final fit under the same scientific design and seeds. It uses eight instead of four one-thread CPU workers. No settings were changed to seek a favorable result; this is not a new blind preregistration. See PROTOCOL.md and protocol.json for full disclosure.\n\n## Install\n\nUse Python 3.12. Install CPU torch2.10.0, numpy2.3.5, jaxtyping0.3.11 and pandas2.2.3. Matplotlib3.10.8 is only needed for delivery figures. Obtain third-party sources separately; they are not bundled. Respect their licenses (torchdde Apache-2.0).\n\n```\npython3 -m venv .venv\n.venv/bin/python -m pip install torch==2.10.0 --index-url https://download.pytorch.org/whl/cpu\n.venv/bin/python -m pip install numpy==2.3.5 jaxtyping==0.3.11 pandas==2.2.3 matplotlib==3.10.8\ngit clone --branch DCE_NDDE https://github.com/onofriosmr/torchdde.git external_torchdde\ngit -C external_torchdde checkout 4888c63dc84e966f6c6553277adc3851a2cca4fc\ngit clone --branch DCE_NDDE https://github.com/onofriosmr/learnable_delays.git external_learnable_delays\ngit -C external_learnable_delays checkout 9b573c4bed5efee9fe9c6b765044dbd506b5bf07\n.venv/bin/python -m pip install --no-deps -e external_torchdde\n```\n\nFrom the source/ folder, run the reserved validation, then the two scientific stages using a new output folder:\n\n```\n.venv/bin/python validate.py\n.venv/bin/python run.py --results independent_results --solver external_torchdde --architecture external_learnable_delays --stage calibration\n.venv/bin/python run.py --results independent_results --solver external_torchdde --architecture external_learnable_delays --stage final\n.venv/bin/python audit.py --results independent_results --architecture external_learnable_delays\n```\n\nAdjust paths to your venv and external checkouts. The run makes a fresh manifest before fitting. Resume by rerunning the same stage with the same directory; code/config/runtime must match. Completed failures are never retried; atomic worker records recover interruptions. `--limit 32` limits new fits in a call, allowing durable saves between chunks. Repeat the final stage until all120 fits complete. A failed calibration cell can make its final comparisons unavailable. The reserved full-workflow smoke uses the same commands with `--mode smoke`, seeds separate from the study and four updates; do not count it as scientific evidence.\n\nFor the delivered results, run audit.py against results/ after installing the exact recorded runtime. The replay checks frozen source and external hashes, data/anchors, checkpoint metrics, calibration selection/lock and paired summaries. It shares architecture/solver code and is a consistency replay, not independent scientific replication. Different hardware/software can change numerical outcomes; new runs get new manifests. Learning curves are sampled update diagnostics, not complete optimizer state.\n\n## Interpretation\n\nThe source method is Monsel et al., DOI https://doi.org/10.1017/dce.2026.10053, accessible preprint https://arxiv.org/html/2410.02843v2. External-C instantiates its released width/depth with a generic MLP using current/delayed state and one scalar delay. The common Euler/direct-backpropagation, known bounds, analytic prehistory/t0=0 and fixed Adam schedules are disclosed adaptations. This is not a replication of the source paper's original training experiment or adjoint algorithm. External-C+anchors adds our loss and is explicitly an ablation.\n\nStructured-C has an additional affine-channel/negative-gain prior, so cross-architecture differences are not solely due to anchors. Final pairs may also use different validation-selected rates. All fields and scalar delays are unknown during fitting; true coefficients/delays are used only in reference generation and postfit scores. Anchor-C uses observed training slopes only and has no forecast score. This constant-delay subcase does not compare unknown state-dependent-delay learning methods. Preserve all unfavorable comparisons and E1–E3's earlier criteria when integrating the manuscript.\n"}
for name,text in source_files.items():
    (root/name).write_text(text)


In [ ]:
subprocess.run([sys.executable,"-m","pip","install","torch==2.10.0","--index-url","https://download.pytorch.org/whl/cpu"],check=True)
subprocess.run([sys.executable,"-m","pip","install","numpy==2.3.5","jaxtyping==0.3.11","pandas==2.2.3"],check=True)
for name,url,commit in [("external_torchdde","https://github.com/onofriosmr/torchdde.git","4888c63dc84e966f6c6553277adc3851a2cca4fc"),("external_learnable_delays","https://github.com/onofriosmr/learnable_delays.git","9b573c4bed5efee9fe9c6b765044dbd506b5bf07")]:
    folder=root/name
    if not folder.exists():
        subprocess.run(["git","clone","--branch","DCE_NDDE",url,str(folder)],check=True)
        subprocess.run(["git","-C",str(folder),"checkout",commit],check=True)
    assert subprocess.check_output(["git","-C",str(folder),"rev-parse","HEAD"],text=True).strip()==commit
subprocess.run([sys.executable,"-m","pip","install","--no-deps","-e",str(root/"external_torchdde")],check=True)


In [ ]:
def run(script,*args):
    subprocess.run([sys.executable,str(root/script),*map(str,args)],cwd=root,check=True)
common=["--solver",root/"external_torchdde","--architecture",root/"external_learnable_delays"]
run("validate.py")
run("run.py","--results",root/"reserved_smoke",*common,"--mode","smoke","--stage","calibration")
run("run.py","--results",root/"reserved_smoke",*common,"--mode","smoke","--stage","final")
run("audit.py","--results",root/"reserved_smoke","--architecture",root/"external_learnable_delays")


In [ ]:
results=root/"independent_results"
run("run.py","--results",results,*common,"--stage","calibration")


In [ ]:
run("run.py","--results",results,*common,"--stage","final")
run("audit.py","--results",results,"--architecture",root/"external_learnable_delays")


In [ ]:
import shutil
archive=shutil.make_archive("/content/ASOC_E4_Independent_Results","zip",root_dir=results)
from google.colab import files
files.download(archive)
